# VGG16 Transfer Learning — Emotion Detection Fine-Tuning

**FER2013 • Selective Fine-Tuning • 7-Class Classification**

This notebook is part of a portfolio demonstrating **transfer learning with VGG16**.  
The goal is to compare pretrained feature extraction with task-specific fine-tuning while keeping the workflow reproducible and easy to inspect on GitHub.

> **Training note:** `EPOCHS = 2` is intentionally used for quick portfolio runs. Increase it later when doing a full experiment.

## Approach

1. Prepare the dataset
2. Load VGG16 pretrained on ImageNet
3. Build a lightweight task-specific classifier
4. Train for a short 2-epoch demonstration
5. Visualize training/validation performance
6. Run a sample prediction

The notebooks are designed to be readable as standalone project artifacts rather than raw classroom/Colab notebooks.

## 1. Configuration

This is the fine-tuning counterpart to the feature-extraction notebook.  
The VGG16 layers from `block5_conv1` onward are trainable, while earlier layers remain frozen.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

SEED = 42
IMAGE_SIZE = (150, 150)
BATCH_SIZE = 32
EPOCHS = 2
DATA_ROOT = Path("fer2013_dataset")

np.random.seed(SEED)
tf.random.set_seed(SEED)

TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"

assert TRAIN_DIR.exists(), f"Training directory not found: {TRAIN_DIR}"
assert TEST_DIR.exists(), f"Test directory not found: {TEST_DIR}"

### Optional: download FER2013 in Google Colab

In [ ]:
# !pip -q install kaggle
# !kaggle datasets download -d msambare/fer2013
# !unzip -q fer2013.zip -d fer2013_dataset

## 2. Load VGG16 and selectively unfreeze deeper layers

In [ ]:
conv_base = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(*IMAGE_SIZE, 3)
)

set_trainable = False

for layer in conv_base.layers:
    if layer.name == "block5_conv1":
        set_trainable = True
    layer.trainable = set_trainable

trainable_layers = [layer.name for layer in conv_base.layers if layer.trainable]
frozen_layers = [layer.name for layer in conv_base.layers if not layer.trainable]

print("Frozen convolutional layers:", len(frozen_layers))
print("Trainable convolutional layers:", len(trainable_layers))
print("Trainable layers:", trainable_layers)

## 3. Build the seven-class classifier

In [ ]:
model = Sequential([
    conv_base,
    Flatten(),
    Dense(256, activation="relu"),
    Dense(7, activation="softmax")
])

model.summary()

## 4. Prepare image generators

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    seed=SEED
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

print("Class mapping:", train_generator.class_indices)

## 5. Fine-tune for 2 epochs

A low learning rate is used to make smaller updates to the pretrained weights.

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

model.compile(
    optimizer=keras.optimizers.RMSprop(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=test_generator,
    callbacks=[early_stopping],
    verbose=1
)

## 6. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"], marker="o", label="Train")
axes[0].plot(history.history["val_accuracy"], marker="o", label="Validation")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"], marker="o", label="Train")
axes[1].plot(history.history["val_loss"], marker="o", label="Validation")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Sample prediction

In [ ]:
class_names = {v: k for k, v in train_generator.class_indices.items()}

sample_path = next(TEST_DIR.glob("*/*.jpg"), None)
if sample_path is None:
    sample_path = next(TEST_DIR.glob("*/*.jpeg"), None)

if sample_path:
    img = load_img(sample_path, target_size=IMAGE_SIZE)
    img_array = img_to_array(img) / 255.0
    probabilities = model.predict(np.expand_dims(img_array, axis=0), verbose=0)[0]

    predicted_index = int(np.argmax(probabilities))
    print("Image:", sample_path)
    print("Predicted emotion:", class_names[predicted_index])
    print(f"Confidence: {probabilities[predicted_index]:.3f}")
else:
    print("No JPG/JPEG sample found in the test directory.")

## Key takeaway

This notebook extends the feature-extraction baseline by adapting the deeper VGG16 representation to facial emotion recognition. The 2-epoch setting is deliberately lightweight for GitHub demonstration and can be increased for a full training experiment.